In [38]:
import pandas as pd
import os
from dotenv import load_dotenv  
import requests
from time import sleep
import time
import json
import requests
from requests.exceptions import JSONDecodeError

In [39]:
load_dotenv()
WEB_MANUFACTURER = os.getenv("WEB_MANUFACTURER")
WEB_MODELS = os.getenv("WEB_MODELS")
WEB_BASE = os.getenv("WEB_BASE")
WEB_VERSIONS = os.getenv("WEB_VERSIONS")
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": f"{WEB_BASE}/comprar-coche/",
    "Origin": f"{WEB_BASE}"
})

In [40]:
url_marcas = f"{WEB_MANUFACTURER}"

In [ ]:
try:
    marcas_json = session.get(url_marcas).json()
    marcas = marcas_json.get('wkda', {})
except Exception as e:
    print(f"Error inicial al obtener marcas: {e}")
    marcas = {}

catalogo = []

for m_id, nombre_marca in marcas.items():
    # Obtener modelos
    url_modelos = f"{WEB_MODELS}{m_id}"
    try:
        resp_modelos = session.get(url_modelos)
        modelos_dict = resp_modelos.json().get('wkda', {})
    except Exception as e:
        print(f"Error en modelos de {nombre_marca}: {e}")
        continue

    # Obtener versiones
    url_versiones = f"{WEB_VERSIONS}/{m_id}/sub-types?manufacturer={m_id}"
    print(url_versiones)
    versiones_data = {}
    
    try:
        resp_versiones = session.get(url_versiones)
        if resp_versiones.status_code == 200:
            versiones_data = resp_versiones.json()
        else:
            print(f"Error {resp_versiones.status_code} en versiones de {nombre_marca}")
    except JSONDecodeError:
        print(f"La web devolvió HTML en vez de JSON para {nombre_marca} (posible bloqueo)")
    except Exception as e:
        print(f"Error inesperado en {nombre_marca}: {e}")

    # Cruzar datos 
    lista_modelos_con_versiones = []

    for mod_id, mod_nombre in modelos_dict.items():
        # Buscamos el ID del modelo dentro del JSON de versiones
        detalle_modelo = versiones_data.get(mod_id, {})
        subtipos = detalle_modelo.get('subtypes', {})
        
        lista_modelos_con_versiones.append({
            "modelo_id": mod_id,
            "modelo_nombre": mod_nombre,
            "versiones": subtipos
        })

    marca_final = {
        "id_marca": m_id,
        "nombre": nombre_marca,
        "modelos": lista_modelos_con_versiones,
        "total_modelos": len(lista_modelos_con_versiones)
    }

    catalogo.append(marca_final)
    
    print(f"Añadida marca {nombre_marca} ({len(lista_modelos_con_versiones)} modelos)")
    sleep(2)

# Guardar
with open('catalogo_final.json', 'w', encoding='utf-8') as f:
    json.dump(catalogo, f, ensure_ascii=False, indent=4)

https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/020/sub-types?manufacturer=020
✅ Añadida marca Abarth (13 modelos)
https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/032/sub-types?manufacturer=032
✅ Añadida marca Aiways (2 modelos)
https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/035/sub-types?manufacturer=035
✅ Añadida marca Aixam (6 modelos)
https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/040/sub-types?manufacturer=040
✅ Añadida marca Alfa Romeo (26 modelos)
https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/042/sub-types?manufacturer=042
✅ Añadida marca Alpina (11 modelos)
https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/043/sub-types?manufacturer=043
✅ Añadida marca Alpine (2 modelos)
https://www.compramostucoche.es/comprar-coche/api/v1/car-types/manufacturer/057/sub-types?manufacturer=057
✅ Añadida marca Aston Martin (9 mod

In [ ]:
# Listas para recolectar los datos
data_marcas = []
data_modelos = []
data_versiones = []

# Contador para IDs únicos de modelos y versiones
id_modelo_relacional = 1
id_version_relacional = 1

for marca in catalogo:
    # 1. Procesar Marcas
    data_marcas.append({
        "marca_id": marca["id_marca"],
        "nombre_marca": marca["nombre"]
    })
    
    for modelo in marca["modelos"]:
        # 2. Procesar Modelos
        current_model_id = id_modelo_relacional
        data_modelos.append({
            "modelo_id": current_model_id,
            "id_web": modelo["modelo_id"],
            "nombre_modelo": modelo["modelo_nombre"],
            "marca_id": marca["id_marca"]
        })
        
        # 3. Procesar Versiones
        for clave_v, nombre_v in modelo["versiones"].items():
            data_versiones.append({
                "version_id": id_version_relacional,
                "modelo_id": current_model_id,
                "clave_motor": clave_v,
                "nombre_completo": nombre_v
            })
            id_version_relacional += 1
            
        id_modelo_relacional += 1

# CSV de Marcas
df_marcas = pd.DataFrame(data_marcas)
df_marcas.to_csv('../datasets/seed_marcas.csv', index=False, encoding='utf-8-sig', sep=';')

# CSV de Modelos
df_modelos = pd.DataFrame(data_modelos)
df_modelos.to_csv('../datasets/seed_modelos.csv', index=False, encoding='utf-8-sig', sep=';')

# CSV de Versiones
df_versiones = pd.DataFrame(data_versiones)
df_versiones.to_csv('../datasets/seed_versiones.csv', index=False, encoding='utf-8-sig', sep=';')

print(f"Archivos generados: {len(data_marcas)} marcas, {len(data_modelos)} modelos y {len(data_versiones)} versiones.")

¡Exportación completada!
Archivos generados: 117 marcas, 1526 modelos y 8559 versiones.
